# Water Dataset Inspection

This notebook loads the generated graph dataset and prints the first sample with its tensor shapes.

Make sure `DATASETS/water_dataset.pt` exists before running this notebook.

In [4]:
! pip install torch torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 21.1 MB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 39.2 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 43.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 30.6 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 52.2 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 36.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 27.5 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 49.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 45.0 MB/s  0:00:026m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 41.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [6]:
import torch
from torch_geometric.data import Data

dataset = torch.load(
    "../DATASETS/water_dataset.pt",
    weights_only=False
)

print(type(dataset))
print(len(dataset))

sample = dataset[0]

print(sample)
print(sample.x.shape)
print(sample.edge_index.shape)
print(sample.y)

/workspaces/water-distribution-networks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'list'>
2500
Data(x=[4, 8], edge_index=[2, 2], y=[1], num_nodes=4)
torch.Size([4, 8])
torch.Size([2, 2])
tensor([0])


## Complete Dataset Analysis

This cell computes global dataset statistics and explains the dataset structure.

- `dataset` is a Python list of `torch_geometric.data.Data` objects.
- `x` is the node feature matrix: each row is one pipe and each column is a pipe feature.
- `edge_index` is the graph connectivity in COO format: each column `[i, j]` means pipe `i` is connected to pipe `j` because they share a network node.
- `y` is the label tensor containing the index of the cracked pipe in this sample.

In [ ]:
from collections import Counter

sample = dataset[0]
print("--- First sample overview ---")
print("Data keys:", sample.keys)
print("x shape:", sample.x.shape, "(num_pipes, num_features)")
print("edge_index shape:", sample.edge_index.shape, "(2, num_edges)")
print("y shape:", sample.y.shape, "value:", sample.y.tolist())
print()
print("Feature vector for pipe 0:", sample.x[0].tolist())
print()
print("First 20 edge pairs (pipe index connections):")
edge_pairs = [tuple(pair) for pair in sample.edge_index.t().tolist()]
print(edge_pairs[:20])
print()
print("Total nodes (pipes) in sample:", sample.x.shape[0])
print("Total directed edges in sample:", sample.edge_index.shape[1])
print("Unique pipe labels in sample (y is pipe index):", set([int(sample.y.item())]))
print()
print("--- Dataset-wide statistics ---")

num_samples = len(dataset)
node_counts = [data.x.shape[0] for data in dataset]
edge_counts = [data.edge_index.shape[1] for data in dataset]
y_values = [int(data.y.item()) for data in dataset]

print("Number of samples:", num_samples)
print("Pipe counts: min=", min(node_counts), ", max=", max(node_counts), ", mean=", sum(node_counts)/len(node_counts))
print("Edge counts: min=", min(edge_counts), ", max=", max(edge_counts), ", mean=", sum(edge_counts)/len(edge_counts))
print("Unique cracked-pipe labels:", len(set(y_values)))
print()
print("Label distribution for cracked pipe indices (top 20):")
for label, count in Counter(y_values).most_common(20):
    print(f"  pipe index {label}: {count}")

print()
print("Example mapping of y to pipe node:")
print("  For this sample, y =", int(sample.y.item()), "means the cracked pipe is node index", int(sample.y.item()))
print("  That node corresponds to sample.x[y] as the cracked pipe's feature vector.")

In [ ]:
# Optional: inspect the feature names and how they map to x columns.
feature_names = [
    "Q1",
    "Q2",
    "Q_leak",
    "Hm",
    "f",
    "Q_EPANET",
    "H_in",
    "H_out"
]
print("Feature names:")
for idx, name in enumerate(feature_names):
    print(f"  x[:, {idx}] = {name}")